# 13. Top/Bottom Content


> **Dataset note:** This project uses a synthetic OTT viewer-behaviour dataset for educational analysis.  
> **Hotstar is used only as the business case/scenario; the data is not claimed to be proprietary Hotstar data.**

**Allowed tools:** Python, NumPy, Pandas, Matplotlib, Seaborn.  
**Not used:** Scikit-learn, Plotly, Power BI, Tableau, machine learning, or statistical libraries beyond NumPy/Pandas.

## Process Performed
I ranked top and bottom episodes, identified highest watch completion and drop-off probability, and built promotion candidates using engagement, completion, and drop-off together.

In [1]:
# PROCESS: Load the CSV into a Pandas DataFrame and verify that the file loaded correctly.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 140)

DATA_PATH = "ott_viewer_dropoff_retention_us_v1.0.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
df.head()

Dataset shape: (33171, 23)


,show_id,title,platform,genre,release_year,season_number,episode_number,episode_duration_min,pacing_score,hook_strength,dialogue_density,visual_intensity,avg_watch_percentage,pause_count,rewind_count,skip_intro,cognitive_load,attention_required,night_watch_safe,drop_off,drop_off_probability,retention_risk,dataset_version
0,66732,Stranger Things,Netflix,Sci-Fi & Fantasy,2016.0,1,1,48,4,5,high,5,39,3,0,0,9,high,0,1,0.649,high,v1.0
1,66732,Stranger Things,Netflix,Sci-Fi & Fantasy,2016.0,1,2,55,5,4,low,8,55,3,3,1,5,medium,0,0,0.473,medium,v1.0
2,66732,Stranger Things,Netflix,Sci-Fi & Fantasy,2016.0,1,3,51,4,8,high,7,46,4,2,0,9,high,0,0,0.583,medium,v1.0
3,66732,Stranger Things,Netflix,Sci-Fi & Fantasy,2016.0,1,4,50,4,7,medium,3,50,4,1,0,7,high,0,0,0.520,medium,v1.0
4,66732,Stranger Things,Netflix,Sci-Fi & Fantasy,2016.0,1,5,52,4,3,low,4,35,3,0,1,7,high,0,1,0.638,high,v1.0


In [2]:
# PROCESS: Create the project-defined engagement score before ranking or grouping content.
# Engagement score based on the formula provided in the project brief.
# Watch percentage is already on a 0-100 scale, while the 1-10 content scores
# are multiplied by 5 before combining them.
df["engagement_score"] = (
    df["avg_watch_percentage"]
    + df["hook_strength"] * 5
    + df["pacing_score"] * 5
    + df["visual_intensity"] * 5
) / 2

df["engagement_score"].describe()

count    33171.000000
mean        70.734135
std         12.023213
min         30.000000
25%         62.500000
50%         71.000000
75%         79.500000
max        117.500000
Name: engagement_score, dtype: float64

In [3]:
# KERNEL-SAFE SETUP
# Keep plotting memory controlled while preserving the analysis.
import gc

MAX_PLOT_ROWS = 5000

def safe_sample(data, n=MAX_PLOT_ROWS, random_state=42):
    """Use the full data when small; otherwise use a reproducible sample for plotting."""
    return data if len(data) <= n else data.sample(n=n, random_state=random_state)

def finish_plot():
    """Render and release the current Matplotlib figure."""
    plt.tight_layout()
    plt.show()
    plt.close()
    gc.collect()

## Top 10 Episodes by Engagement

In [4]:
# PROCESS: Create the project-defined engagement score before ranking or grouping content.
columns = [
    "title", "season_number", "episode_number", "genre",
    "avg_watch_percentage", "drop_off_probability",
    "retention_risk", "engagement_score"
]

top10 = df.nlargest(10, "engagement_score")[columns]
display(top10)

,title,season_number,episode_number,genre,avg_watch_percentage,drop_off_probability,retention_risk,engagement_score
16149,30 Rock,1,1,Comedy,100,0.120,low,117.5
1840,Young Sheldon,1,1,Comedy,99,0.154,low,114.5
13312,Buffy the Vampire Slayer,1,1,Sci-Fi & Fantasy,98,0.169,low,111.5
7315,The Late Show with Stephen Colbert,1,12,Comedy,94,0.187,low,109.5
1408,Tagesschau,1,1,News,86,0.243,low,108.0
1198,Arrow,1,1,Crime,85,0.238,low,107.5
27533,Good Mythical Morning,1,50,Comedy,89,0.219,low,107.0
33099,The Cosby Show,1,6,Comedy,89,0.209,low,107.0
197,Law & Order,1,1,Crime,98,0.219,low,106.5
2342,Running Man,1,244,Comedy,88,0.244,low,106.5


## Bottom 10 Episodes by Engagement

In [5]:
# PROCESS: Create the project-defined engagement score before ranking or grouping content.
bottom10 = df.nsmallest(10, "engagement_score")[columns]
display(bottom10)

,title,season_number,episode_number,genre,avg_watch_percentage,drop_off_probability,retention_risk,engagement_score
21243,Dreams of Liberty,1,274,Drama,15,0.808,high,30.0
14714,A Kindred Spirit,1,575,Drama,13,0.806,high,31.5
20662,6 of Us,1,4,Drama,20,0.765,high,32.5
28492,Ghum Hai Kisikey Pyaar Meiin,1,73,Drama,20,0.755,high,32.5
28319,The Closer,1,6,Crime,17,0.788,high,33.5
20583,AIBOU: Tokyo Detective Duo,1,10,Drama,23,0.742,high,34.0
21033,Dreams of Liberty,1,64,Drama,13,0.796,high,34.0
28977,Ghum Hai Kisikey Pyaar Meiin,1,558,Drama,23,0.742,high,34.0
5007,Grimm,1,7,Drama,19,0.790,high,34.5
14656,A Kindred Spirit,1,517,Drama,24,0.737,high,34.5


## Highest Watch Percentage

In [6]:
# PROCESS: Create the project-defined engagement score before ranking or grouping content.
highest_watch = (
    df.nlargest(10, "avg_watch_percentage")
    [["title", "season_number", "episode_number", "genre",
      "avg_watch_percentage", "engagement_score"]]
)
display(highest_watch)

,title,season_number,episode_number,genre,avg_watch_percentage,engagement_score
16149,30 Rock,1,1,Comedy,100,117.5
25672,2 Broke Girls,1,1,Comedy,100,100.0
1840,Young Sheldon,1,1,Comedy,99,114.5
197,Law & Order,1,1,Crime,98,106.5
13312,Buffy the Vampire Slayer,1,1,Sci-Fi & Fantasy,98,111.5
3808,Ninja Boy Rantaro,1,45,Comedy,94,102.0
7315,The Late Show with Stephen Colbert,1,12,Comedy,94,109.5
2490,Running Man,1,392,Comedy,93,104.0
23578,Will & Grace,1,10,Comedy,93,104.0
32678,CID,1,1132,Action & Adventure,93,94.0


## Highest Drop-Off Probability

In [7]:
# PROCESS: Create the project-defined engagement score before ranking or grouping content.
highest_dropoff = (
    df.nlargest(10, "drop_off_probability")
    [["title", "season_number", "episode_number", "genre",
      "avg_watch_percentage", "drop_off_probability",
      "retention_risk", "engagement_score"]]
)
display(highest_dropoff)

,title,season_number,episode_number,genre,avg_watch_percentage,drop_off_probability,retention_risk,engagement_score
20822,Blood Flowers,1,42,Drama,21,0.831,high,38.0
6280,Doraemon,1,1259,Action & Adventure,18,0.824,high,36.5
21860,Pleasant Goat and Big Big Wolf,1,510,Animation,23,0.822,high,44.0
15252,A Kindred Spirit,1,1113,Drama,21,0.821,high,40.5
31170,Show! Music Core,1,762,Reality,24,0.817,high,37.0
14403,A Kindred Spirit,1,264,Drama,20,0.815,high,47.5
31394,Station 19,1,5,Drama,16,0.813,high,38.0
25255,This Is Us,1,6,Drama,23,0.811,high,39.0
32169,CID,1,623,Action & Adventure,19,0.809,high,44.5
10680,Sazae-san,1,2160,Animation,15,0.808,high,37.5


## Promotion Candidates

In [8]:
# PROCESS: Create the project-defined engagement score before ranking or grouping content.
# Promotion candidates combine high engagement, high completion,
# and comparatively low drop-off probability.
eng_cut = df["engagement_score"].quantile(0.90)
watch_cut = df["avg_watch_percentage"].quantile(0.75)
drop_cut = df["drop_off_probability"].quantile(0.25)

promotion_candidates = (
    df[
        (df["engagement_score"] >= eng_cut) &
        (df["avg_watch_percentage"] >= watch_cut) &
        (df["drop_off_probability"] <= drop_cut)
    ]
    [columns]
    .sort_values(["engagement_score", "avg_watch_percentage"],
                 ascending=False)
)

print("Promotion thresholds")
print("Engagement >=", round(eng_cut, 2))
print("Watch % >=", round(watch_cut, 2))
print("Drop-off probability <=", round(drop_cut, 3))
display(promotion_candidates.head(20))

Promotion thresholds
Engagement >= 86.5
Watch % >= 66.0
Drop-off probability <= 0.4


,title,season_number,episode_number,genre,avg_watch_percentage,drop_off_probability,retention_risk,engagement_score
16149,30 Rock,1,1,Comedy,100,0.120,low,117.5
1840,Young Sheldon,1,1,Comedy,99,0.154,low,114.5
13312,Buffy the Vampire Slayer,1,1,Sci-Fi & Fantasy,98,0.169,low,111.5
7315,The Late Show with Stephen Colbert,1,12,Comedy,94,0.187,low,109.5
1408,Tagesschau,1,1,News,86,0.243,low,108.0
1198,Arrow,1,1,Crime,85,0.238,low,107.5
27533,Good Mythical Morning,1,50,Comedy,89,0.219,low,107.0
33099,The Cosby Show,1,6,Comedy,89,0.209,low,107.0
197,Law & Order,1,1,Crime,98,0.219,low,106.5
2342,Running Man,1,244,Comedy,88,0.244,low,106.5


### Interpretation Note
The outputs above describe patterns in the supplied **synthetic episode-level dataset**. They should support business hypotheses and content investigation, not be presented as causal proof or proprietary Hotstar findings.

## Decision Rule

Top/bottom rankings should not be interpreted from engagement alone. Promotion candidates are strongest when they combine **high engagement + high watch completion + low drop-off probability**. Investigation candidates should show the opposite pattern and/or High retention risk.

## Reliability-Aware Show Ranking

Episode rankings can be shown directly, but show-level rankings should exclude extremely small samples.  
Here, shows need at least 10 episode-level observations before being ranked.

In [9]:
df["engagement_index"] = (
    df["avg_watch_percentage"]
    + df["hook_strength"] * 5
    + df["pacing_score"] * 5
    + df["visual_intensity"] * 5
) / 2

show_rank = (
    df.groupby(["show_id", "title"])
    .agg(
        episodes=("episode_number", "count"),
        avg_engagement=("engagement_index", "mean"),
        median_engagement=("engagement_index", "median"),
        avg_watch=("avg_watch_percentage", "mean"),
        avg_drop_probability=("drop_off_probability", "mean")
    )
    .reset_index()
)

reliable_show_rank = show_rank[
    show_rank["episodes"] >= 10
]

print("Top reliable shows:")
display(
    reliable_show_rank
    .sort_values("avg_engagement", ascending=False)
    .head(10)
    .round(2)
)

print("\nBottom reliable shows:")
display(
    reliable_show_rank
    .sort_values("avg_engagement", ascending=True)
    .head(10)
    .round(2)
)

Top reliable shows:


,show_id,title,episodes,avg_engagement,median_engagement,avg_watch,avg_drop_probability
32,1215,Californication,12,83.67,84.50,71.92,0.36
132,4546,Curb Your Enthusiasm,10,81.85,83.50,68.70,0.37
140,4608,30 Rock,21,81.07,80.50,65.95,0.41
303,71728,Young Sheldon,22,81.00,81.50,69.05,0.38
30,1100,How I Met Your Mother,22,80.89,80.75,66.55,0.40
130,4454,Will & Grace,22,80.75,83.00,68.32,0.39
372,112888,True Beauty,16,80.03,79.25,66.31,0.40
129,4419,Real Time with Bill Maher,20,80.00,79.25,66.25,0.40
211,39340,2 Broke Girls,24,79.98,79.50,66.62,0.40
237,48891,Brooklyn Nine-Nine,22,79.50,78.00,67.18,0.40



Bottom reliable shows:


,show_id,title,episodes,avg_engagement,median_engagement,avg_watch,avg_drop_probability
475,281013,Dynamite Kiss,14,60.79,61.50,45.14,0.59
423,222766,The Day of the Jackal,10,61.30,60.50,46.60,0.55
466,274556,Far Away,28,61.57,62.75,48.14,0.54
263,61511,Father Brown,10,61.70,56.50,49.90,0.55
210,39272,Once Upon a Time,22,62.00,59.75,49.23,0.53
457,259730,The Accident,10,62.45,63.50,48.90,0.54
315,76331,Succession,10,62.90,61.25,51.30,0.52
374,113988,DAHMER - Monster: The Jeffrey Dahmer Story,10,63.05,60.50,49.60,0.53
428,226637,High Potential,13,63.12,62.00,49.31,0.55
62,1435,The Good Wife,23,63.50,65.00,49.83,0.53


In [10]:
# KERNEL-SAFE CLEANUP
plt.close('all')
gc.collect()
print('Notebook cleanup complete.')

Notebook cleanup complete.
